In [2]:
import matplotlib.pyplot as plt
import numpy as np

def plot_image(image, title=''):
    plt.imshow(image)
    plt.axis('off')
    plt.title(title, size=20)

def plot_image2(image, title=''):
    imager = image[...,::-1]
    plt.imshow(imager)
    plt.axis('off')
    plt.title(title, size=20)

In [ ]:
from google.colab.patches import cv2_imshow
import cv2

def cornerHarris(img, ksize, k):
    dx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize)   # 수직 소벨 마스크
    dy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize)   # 수평 소벨 마스크

    a = cv2.GaussianBlur(dx * dx, (5, 5), 0)                     # 가우시안 블러링 수행
    b = cv2.GaussianBlur(dy * dy, (5, 5), 0)
    c = cv2.GaussianBlur(dx * dy, (5, 5), 0)

    corner = (a * b - c * c) - k * (a + b) ** 2        # 코너 응답 함수 계산 -행렬 연산 적용
    return corner

def drawCorner(corner, image, thresh):
    corner = cv2.normalize(corner, 0, 300, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    pts = np.where(corner > thresh)
    h, w = corner.shape
    corners = []
    for j, i in np.transpose(pts):
        if 0 < i < h-1 and 0< j < w-1 :
            neighbor = corner[i-1:i+2, j-1:j+2].flatten()
            max = np.max(neighbor[1::2])
            if corner[i, j] >= max: corners.append((j, i))

    for pt in corners:
        pt = (pt[1], pt[0])
        #cv2.circle(image, pt, 3, (0, 0, 255), -1)    # 좌표 표시
        cv2.circle(image, pt, 10, (0, 0, 255), -1)    # 좌표 표시
    print("임계값: %2d , 코너 개수: %2d" %(thresh, len(corners)) )
    return image

image = cv2.imread('chessboard.png', cv2.IMREAD_COLOR)
#image = cv2.imread('harris.jpg', cv2.IMREAD_COLOR)

blockSize, apertureSize = 4, 3                                # 소벨 마스크 크기
k, thresh = 0.04, 2                                           # 코너 응답 임계값

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
corner1 = cornerHarris(gray, apertureSize, k)                # 사용자 정의 함수
corner2 = cv2.cornerHarris(gray, blockSize, apertureSize, k) # OpenCV 제공 함수

thresh = 7.5
img1 = drawCorner(corner1, np.copy(image), thresh)
img2 = drawCorner(corner2, np.copy(image), thresh)

plt.figure(figsize=(20,10))
plt.gray()
plt.subplot(311), plot_image(gray, 'Source')
plt.subplot(312), plot_image2(img1, 'User Harris')
plt.subplot(313), plot_image2(img2, 'OpenCV Harris')
plt.show()